# Defining a .geo file and generating a mesh

In this section we discuss how to create a geometry file (.geo), with the geometry file being interpreted by Netgen (the meshing tool in NGSolve) we first explain how to create shapes and geometries using a .geo file.

Due to Netgen being the underlying interpreter of the .geo file we direct the user to the relevant Constructive Solid Geometry (CSG) documentation available at [http://netgen-mesher.sourceforge.net/docs/ng4.pdf](http://netgen-mesher.sourceforge.net/docs/ng4.pdf) and the large number of example geo files avaliable on the Netgen github page ([https://github.com/NGSolve/netgen/tree/master/tutorials](https://github.com/NGSolve/netgen/tree/master/tutorials)).

We begin by setting up the required packages

In [1]:
from time import time
import numpy as np
from matplotlib import pyplot as plt
import os
root_dir = os.path.dirname(os.getcwd())
os.chdir(root_dir)
print(os.getcwd())

/Users/cgledger/Coding/MPT-Calculator_v2/MPT-Calculator


## Controlling the  material parmeter of  a conducting sphere using a .geo file
So far we haved considered the MPT characterisation of a conducting sphere with $\sigma_* =1  \times 10^6 $ S/m, $\mu_r=1$ and radius $\alpha =1 \times 10^{-3}$ m by constructing a coarse unstructured tetrahedral discretisation of a unit sphere set in truncated unbounded region in the form of a larger sphere of radius 200 units using order $p=2$ elements. Note the physical object is obtained by scaling the non-dimensional computational domain by $\alpha$. Now wish to control the materials of the sphere in more detail.

In the folder `Geofiles/` there are a series of pre-setup object descriptions including `sphere.geo` that contains the following

```
algebraic3d


solid sphout = sphere (0, 0, 0; 200);
solid sphin = sphere (0, 0, 0; 1) -maxh=0.5;

solid rest = sphout and not sphin;

tlo rest -transparent -col=[0,0,1];#air
tlo sphin -col=[1,0,0];#sphere -mur=1 -sig=1e6

```

In [4]:
from netgen.csg import *
from ngsolve import *
from ngsolve.webgui import Draw
geo = CSGeometry(r'GeoFiles/sphere.geo')
mesh = geo.GenerateMesh()
ngmesh = Mesh(mesh)
Draw(ngmesh)

TypeError: encodeData() got an unexpected keyword argument 'dtype'

```
algebraic3d
```
is the top level command that every .geo file requires before any non commented lines.

```
solid sphout = sphere (0, 0, 0; 200);
```
We can define solids using the $\texttt{solid}$ and a primitive sphere using the $\texttt{sphere}$ keyword
```
solid sphin = sphere (0, 0, 0; 1) -maxh=0.5;
```
the $\texttt{maxh}$ tag can be used to directly specify the maximum element size inside the associated object. For example, here we set the maximum element size inside the inner sphere to be 0.5.

The keywords $\texttt{and}$ and $\texttt{or}$ denote the intersection and union respectivlty of the two solids, so the line
```
solid rest = sphout and not sphin;
```
defines a solid that contains only the region that is occupied by the outer sphere and not the inner sphere.

Finally, we wish to define which objects will be meshed. This is done using the $\texttt{tlo}$ tag.
```
tlo rest -transparent -col=[0,0,1];#air
tlo sphin -col=[1,0,0];#sphere -mur=1 -sig=1e6
```
The $\texttt{#}$ represents a comment and is not used by Netgen. The object names, $\texttt{-mur}$, and $\texttt{-sig}$ are used by MPT-Calculator to assign material properites to each object.

The $\texttt{#air}$ is used to denote vacuum permeability and conductivity.;

I THINK WE SHOULD GIVE ILLUSTRATIVE EXAMPLES THAT ILLUSTRATE WHAT HAPPENS WHEN YOU CHANGE MATERIALS HERE

## Other Primitives

LIST OF OTHER PRIMITIVES? INCLUDE LINKS TO WHERE THESE ARE DOCUMENTED. WHAT OTHER EXAMPLES ARE INCLUDED IN THE GEOFILES FOLDER? 

## Earring

For a more complex object, we work in a very similar way. i.e. defining object primatives and constructing other shapes via unions and intersections.
In this example, we build a simple earring, consisting of spherical end, a cylindrical post, and a flat cylindric back.

#### Sphere
A sphere is defined by an origin $(x,y,z)$ and a radius $r$
```
sphere(x, y, z; r)
```

A cylinder in the .geo format is assumed to be infinitly long. To construct a finite cylinder, we need to calculate the intersection with a plane.


```
algebraic3d

#Outer domain
solid outer_box = orthobrick (-1000, -1000, -1000; 1000, 1000, 1000);

#Object parts
solid ball = sphere(6,0,0;1);
solid bar = cylinder(0,0,0;10,0,0;0.5)
	and plane(0,0,0;-1,0,0)
	and plane(6,0,0;1,0,0);
solid end = cylinder(-1,0,0;1,0,0;1.2)
	and plane(0,0,0;1,0,0)
	and plane(-0.5,0,0;-1,0,0);

#Combining parts
solid earing = ball or bar or end -maxh=0.25;
solid rest = outer_box and not earing;

#Defining top level objects
tlo rest -transparent -col=[0,0,1];#air
tlo earing -col=[1,0,0];#earring -mur=1 -sig=4.1E+07
```

In [5]:
geo = CSGeometry(r'GeoFiles/Earring.geo')
mesh = geo.GenerateMesh()
ngmesh = Mesh(mesh)
Draw(ngmesh)

TypeError: encodeData() got an unexpected keyword argument 'dtype'

In [ ]:
INCLUDE AN EXAMPLE OF HOW THIS IS COMPUTED. INCLUDE RESULTS EITHER BY LOADING FROM THE FILE OR BY RUNNING LIVE